In [19]:
import re
import numpy as np
from scipy.io import loadmat


def _to_numpy(x):
    """Convert MATLAB numeric arrays to NumPy arrays."""
    if isinstance(x, np.ndarray):
        return x
    return np.array(x)


def _clean_mpc_struct(mpc):
    """
    Convert a MATLAB MATPOWER struct into a clean Python dictionary.
    This works well when scipy.loadmat(..., simplify_cells=True) is used.
    """
    if not isinstance(mpc, dict):
        return mpc

    clean = {}

    for key, value in mpc.items():
        if key.startswith("__"):
            continue

        if key in ["baseMVA", "bus", "gen", "branch", "gencost"]:
            clean[key] = _to_numpy(value)
        else:
            clean[key] = value

    return clean


def load_dplib_case(mat_file):
    """
    Load a DPLib distributed case from a .mat file.

    Parameters
    ----------
    mat_file : str
        Path to the DPLib .mat file, for example:
        'pglib_opf_case4917_goc_10regions.mat'

    Returns
    -------
    dplib : dict
        Dictionary containing:
            - filename
            - num_regions
            - regions
            - raw
            - extra
    """

    data = loadmat(
        mat_file,
        simplify_cells=True,
        squeeze_me=True,
        struct_as_record=False
    )

    dplib = {}

    # Original centralized MATPOWER case name
    dplib["filename"] = data.get("filename", None)

    # Number of regions
    dplib["num_regions"] = data.get("num_regions", None)

    # Store all regional MATPOWER cases here
    dplib["regions"] = {}

    # Store non-region metadata here
    dplib["extra"] = {}

    for key, value in data.items():

        if key.startswith("__"):
            continue

        # Regional cases usually have names like:
        # mpc_regionR1, mpc_regionR2, ...
        match = re.match(r"mpc_regionR(\d+)", key)

        if match:
            region_id = int(match.group(1))
            dplib["regions"][region_id] = _clean_mpc_struct(value)
        else:
            dplib["extra"][key] = value

    # Keep raw loaded data too, in case you need something later
    dplib["raw"] = data

    return dplib

In [20]:
dplib_case = load_dplib_case("pglib_opf_case4917_goc_10regions.mat")

print("Original case:", dplib_case["filename"])
print("Number of regions:", dplib_case["num_regions"])

for r, mpc in dplib_case["regions"].items():
    print(f"\nRegion {r}")
    print("Base MVA:", mpc["baseMVA"])
    print("Bus matrix shape:", mpc["bus"].shape)
    print("Gen matrix shape:", mpc["gen"].shape)
    print("Branch matrix shape:", mpc["branch"].shape)

Original case: pglib_opf_case4917_goc
Number of regions: 10

Region 1
Base MVA: 100
Bus matrix shape: (438, 13)
Gen matrix shape: (67, 10)
Branch matrix shape: (490, 13)

Region 2
Base MVA: 100
Bus matrix shape: (614, 13)
Gen matrix shape: (131, 10)
Branch matrix shape: (853, 13)

Region 3
Base MVA: 100
Bus matrix shape: (477, 13)
Gen matrix shape: (189, 10)
Branch matrix shape: (612, 13)

Region 4
Base MVA: 100
Bus matrix shape: (580, 13)
Gen matrix shape: (162, 10)
Branch matrix shape: (761, 13)

Region 5
Base MVA: 100
Bus matrix shape: (596, 13)
Gen matrix shape: (73, 10)
Branch matrix shape: (795, 13)

Region 6
Base MVA: 100
Bus matrix shape: (552, 13)
Gen matrix shape: (192, 10)
Branch matrix shape: (804, 13)

Region 7
Base MVA: 100
Bus matrix shape: (510, 13)
Gen matrix shape: (140, 10)
Branch matrix shape: (725, 13)

Region 8
Base MVA: 100
Bus matrix shape: (293, 13)
Gen matrix shape: (155, 10)
Branch matrix shape: (371, 13)

Region 9
Base MVA: 100
Bus matrix shape: (478, 13)
Ge